# AdaptiveMath-AI — Dataset Acquisition and Leakage-Controlled Research Design

**Notebook 01 of 06**  
**Task:** prequential binary prediction of the next Eedi mathematics response (`IsCorrect`).

The notebook downloads the official Eedi / NeurIPS 2020 archives, preserves the source files, validates their structure and data quality, and fixes the evaluation protocol before model fitting.

## 1. Reproducible environment and project boundaries

The random seed, repository paths and available compute resources are recorded before loading the interaction archive. No outcome data are accessed in this step.

In [1]:
from pathlib import Path
import os, sys, json, math, time, hashlib, zipfile, platform, shutil, textwrap, warnings
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
import psutil

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 19,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 20,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 350,
    "figure.dpi": 350,
})
PALETTE = {
    "blue": "#2F6B8F", "orange": "#C77C2B", "green": "#4F7F3A",
    "purple": "#6D5A8D", "red": "#B45A55", "gray": "#6E7378",
    "light_gray": "#E7EAED", "teal": "#4A8C8A", "gold": "#C9A227",
    "black": "#222222", "white": "#FFFFFF",
}
SEED = 20260713
np.random.seed(SEED)

cwd = Path.cwd().resolve()
root_candidates = [cwd, *cwd.parents]
ROOT = next((p for p in root_candidates if (p / "notebooks").is_dir() and (p / "README.md").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or the notebooks directory.")
requested_dirs = [
    "dataset/raw/archives", "dataset/raw/extracted", "dataset/processed",
    "artifacts", "figures"
]
for rel in requested_dirs:
    (ROOT / rel).mkdir(parents=True, exist_ok=True)
assert (ROOT / "notebooks").is_dir()

try:
    import torch
    mps_available = bool(torch.backends.mps.is_available())
    torch_version = torch.__version__
except Exception:
    mps_available, torch_version = False, "unavailable"
vm = psutil.virtual_memory()
disk = shutil.disk_usage(ROOT)
environment = {
    "project_root": str(ROOT), "python": sys.version.split()[0], "platform": platform.platform(),
    "machine": platform.machine(), "cpu_count_logical": psutil.cpu_count(logical=True),
    "cpu_count_physical": psutil.cpu_count(logical=False), "available_ram_gib": vm.available / 2**30,
    "total_ram_gib": vm.total / 2**30, "free_disk_gib": disk.free / 2**30,
    "torch_version": torch_version, "mps_available": mps_available, "seed": SEED,
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat()
}
display(pd.DataFrame([environment]).T.rename(columns={0:"value"}))
print(f"Project root: {ROOT}")
print("Memory policy: chunk large CSVs; materialize only typed columns needed for each validation.")

,value
project_root,/Users/talgatazykanov/Desktop/Science works/Ma...
python,3.11.15
platform,macOS-26.6-arm64-arm-64bit
machine,arm64
cpu_count_logical,10
cpu_count_physical,10
available_ram_gib,3.099823
total_ram_gib,16.0
free_disk_gib,280.951271
torch_version,2.11.0


Project root: <repository_root>
Memory policy: chunk large CSVs; materialize only typed columns needed for each validation.


### Figure helper contract

Freeze a consistent, scientific-oriented visual contract early. Define reusable helpers locally in this notebook, including explicit labels and unclipped high-resolution export.

Leakage control: Presentation functions never access target labels independently.

In [2]:
def save_and_show_figure(fig, filename):
    path = ROOT / "figures" / filename
    fig.savefig(path, dpi=350, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved figure: {path}")
    return path

def annotate_bars(ax, fmt="{:.3f}"):
    for patch in ax.patches:
        value = patch.get_width() if patch.get_width() != 0 else patch.get_height()
        if patch.get_width() != 0:
            ax.text(patch.get_x()+patch.get_width(), patch.get_y()+patch.get_height()/2,
                    " " + fmt.format(value), va="center", ha="left", clip_on=False)
        else:
            ax.text(patch.get_x()+patch.get_width()/2, patch.get_height(),
                    fmt.format(value), va="bottom", ha="center", clip_on=False)

def wrap_labels(labels, width=28):
    return ["\n".join(textwrap.wrap(str(label), width=width)) for label in labels]

def plot_horizontal_metric_ranking(frame, label_col, metric_col, color=PALETTE["blue"]):
    plot_df = frame.sort_values(metric_col)
    fig, ax = plt.subplots(figsize=(13, max(6, 0.72*len(plot_df))))
    ax.barh(wrap_labels(plot_df[label_col]), plot_df[metric_col], color=color, edgecolor=PALETTE["black"])
    ax.grid(axis="x", color=PALETTE["light_gray"], linewidth=0.8)
    annotate_bars(ax)
    return fig, ax

def place_legend_below(ax, ncol=3):
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=ncol, frameon=False)

print("Defined notebook-local figure helpers: save_and_show_figure, annotate_bars, wrap_labels, plot_horizontal_metric_ranking, place_legend_below")

Defined notebook-local figure helpers: save_and_show_figure, annotate_bars, wrap_labels, plot_horizontal_metric_ranking, place_legend_below


## 2. Streamed official archive download and validation

Acquire the exact official evidence used by all later notebooks. Stream with retries, record HTTP status and byte counts, calculate SHA-256, validate ZIP central directories, and avoid overwriting valid archives.

Leakage control: Both data and starter kit are treated as immutable raw inputs; labels are not used for model selection here.

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ARCHIVE_SPECS = [
    ("https://dqanonymousdata.blob.core.windows.net/neurips-public/data.zip", "eedi_neurips_2020_data.zip"),
    ("https://dqanonymousdata.blob.core.windows.net/neurips-public/starter_kit.zip", "eedi_neurips_2020_starter_kit.zip"),
]

def sha256_file(path, block_size=8*1024*1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def valid_zip(path):
    if not path.exists() or path.stat().st_size == 0:
        return False, "missing_or_empty"
    try:
        with zipfile.ZipFile(path) as archive:
            bad = archive.testzip()
            return bad is None, "ok" if bad is None else f"bad_member:{bad}"
    except Exception as exc:
        return False, f"{type(exc).__name__}:{exc}"

def stream_download(url, destination):
    existing_ok, existing_note = valid_zip(destination)
    if existing_ok:
        return {"status":"existing_valid", "http_status":None, "bytes":destination.stat().st_size, "notes":existing_note}
    retry = Retry(total=5, connect=5, read=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504), allowed_methods=("GET",))
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    partial = destination.with_suffix(destination.suffix + ".partial")
    if partial.exists(): partial.unlink()
    try:
        with session.get(url, stream=True, timeout=(30, 180)) as response:
            status = response.status_code
            response.raise_for_status()
            downloaded = 0
            with partial.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=8*1024*1024):
                    if chunk:
                        handle.write(chunk); downloaded += len(chunk)
            partial.replace(destination)
        ok, note = valid_zip(destination)
        if not ok:
            raise RuntimeError(f"ZIP validation failed: {note}")
        return {"status":"downloaded", "http_status":status, "bytes":downloaded, "notes":note}
    except Exception as exc:
        if partial.exists(): partial.unlink()
        existing_ok, existing_note = valid_zip(destination)
        if existing_ok:
            return {"status":"existing_valid_after_download_failure", "http_status":None,
                    "bytes":destination.stat().st_size, "notes":f"download_error={exc}; {existing_note}"}
        return {"status":"failed", "http_status":None, "bytes":0, "notes":f"{type(exc).__name__}: {exc}"}

manifest_rows=[]
for url, filename in ARCHIVE_SPECS:
    path = ROOT / "dataset/raw/archives" / filename
    started = datetime.now(timezone.utc).isoformat()
    result = stream_download(url, path)
    ok, zip_note = valid_zip(path)
    manifest_rows.append({
        "source_url":url, "local_path":str(path.relative_to(ROOT)),
        "download_status":result["status"], "http_status":result["http_status"],
        "file_size_bytes":path.stat().st_size if path.exists() else 0,
        "sha256":sha256_file(path) if ok else "", "extraction_status":"pending",
        "timestamp":started, "notes":result["notes"]
    })
    print(f"{filename}: HTTP={result['http_status']} status={result['status']} bytes={result['bytes']:,} ZIP={zip_note}")
manifest = pd.DataFrame(manifest_rows)
manifest_path = ROOT / "artifacts/download_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(manifest)
print(f"Saved table: {manifest_path}")

if (manifest.download_status == "failed").any():
    failure_path = ROOT / "artifacts/dataset_download_failure.log"
    failure_path.write_text("Dataset download failed.\n\n" + manifest.to_csv(index=False), encoding="utf-8")
    print(f"Saved diagnostic log: {failure_path}")
    raise RuntimeError("A required official archive is unavailable and no valid existing archive could be used.")

eedi_neurips_2020_data.zip: HTTP=None status=existing_valid bytes=656,787,242 ZIP=ok


eedi_neurips_2020_starter_kit.zip: HTTP=None status=existing_valid bytes=79,651,393 ZIP=ok


,source_url,local_path,download_status,http_status,file_size_bytes,sha256,extraction_status,timestamp,notes
0,https://dqanonymousdata.blob.core.windows.net/...,dataset/raw/archives/eedi_neurips_2020_data.zip,existing_valid,None,656787242,c7f01672360f1adeb3cf9507d72455d7be035bf897e4a1...,pending,2026-07-31T13:47:48.897020+00:00,ok
1,https://dqanonymousdata.blob.core.windows.net/...,dataset/raw/archives/eedi_neurips_2020_starter_ki...,existing_valid,None,79651393,614716afb05e25b4decf81abc6613a692e921521b41e08...,pending,2026-07-31T13:48:00.091740+00:00,ok


Saved table: <repository_root>/artifacts/download_manifest.csv


## 3. Lossless extraction and recursive file discovery

Preserve the raw archive contents and identify actual paths rather than assuming the published layout. Validate safe ZIP member paths, copy bytes without modifying content, skip already-valid same-size files, and recursively inventory extracted files.

Leakage control: No columns are transformed or renamed in `dataset/raw/`.

In [4]:
def safe_extract(archive_path, destination):
    extracted, skipped = 0, 0
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"Unsafe ZIP member path: {info.filename}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists() and target.stat().st_size == info.file_size:
                skipped += 1; continue
            with archive.open(info, "r") as source, target.open("wb") as sink:
                shutil.copyfileobj(source, sink, length=8*1024*1024)
            extracted += 1
    return extracted, skipped

for idx, row in manifest.iterrows():
    archive_path = ROOT / row.local_path
    extracted, skipped = safe_extract(archive_path, ROOT / "dataset/raw/extracted")
    manifest.loc[idx, "extraction_status"] = "complete"
    manifest.loc[idx, "notes"] = f"{row.notes}; extracted_files={extracted}; existing_files_skipped={skipped}"
    print(f"{archive_path.name}: extracted={extracted}, existing_same_size_skipped={skipped}")
manifest.to_csv(manifest_path, index=False)

all_raw_files = sorted(p for p in (ROOT/"dataset/raw/extracted").rglob("*") if p.is_file())
print(f"Discovered {len(all_raw_files)} extracted files.")
display(pd.DataFrame({
    "relative_path":[str(p.relative_to(ROOT)) for p in all_raw_files],
    "size_mib":[round(p.stat().st_size/2**20, 3) for p in all_raw_files]
}).head(30))
print(f"Updated table: {manifest_path}")

eedi_neurips_2020_data.zip: extracted=0, existing_same_size_skipped=1940
eedi_neurips_2020_starter_kit.zip: extracted=0, existing_same_size_skipped=52


Discovered 1992 extracted files.


,relative_path,size_mib
0,dataset/raw/extracted/__MACOSX/._data,0.0
1,dataset/raw/extracted/__MACOSX/._starter_kit,0.0
2,dataset/raw/extracted/__MACOSX/data/._.DS_Store,0.0
3,dataset/raw/extracted/__MACOSX/data/._images,0.0
4,dataset/raw/extracted/__MACOSX/data/._metadata,0.0
5,dataset/raw/extracted/__MACOSX/data/._train_data,0.0
6,dataset/raw/extracted/__MACOSX/data/images/._0.jpg,0.0
7,dataset/raw/extracted/__MACOSX/data/images/._1.jpg,0.0
8,dataset/raw/extracted/__MACOSX/data/images/._10.jpg,0.0
9,dataset/raw/extracted/__MACOSX/data/images/._100.jpg,0.0


Updated table: <repository_root>/artifacts/download_manifest.csv


## 4. File sizes, schemas, and memory estimates

Establish which files can be materialized safely and detect actual column spelling/capitalization. Count CSV lines by buffered binary scans, infer schemas from bounded samples, and estimate in-memory size from sample expansion.

Leakage control: Sampling is used only for schema/memory validation, not model fitting or feature selection.

In [5]:
def buffered_line_count(path, block_size=16*1024*1024):
    count = 0
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            count += block.count(b"\n")
    return max(count - 1, 0)

file_summary=[]; schema_summary=[]
for path in all_raw_files:
    suffix = path.suffix.lower()
    row_count = None; n_cols = None; memory_estimate = None; read_note="not_tabular"
    if suffix == ".csv":
        try:
            sample = pd.read_csv(path, nrows=5000, low_memory=False)
            row_count = buffered_line_count(path)
            n_cols = sample.shape[1]
            per_row = sample.memory_usage(deep=True).sum()/max(len(sample),1)
            memory_estimate = int(per_row*row_count)
            read_note="sampled"
            for column in sample.columns:
                schema_summary.append({
                    "file":str(path.relative_to(ROOT)), "column":column,
                    "sample_dtype":str(sample[column].dtype),
                    "sample_non_null":int(sample[column].notna().sum()),
                    "sample_unique":int(sample[column].nunique(dropna=True)),
                    "example_values":" | ".join(map(str, sample[column].dropna().astype(str).head(3).tolist()))
                })
        except Exception as exc:
            read_note=f"read_error:{type(exc).__name__}:{exc}"
    file_summary.append({
        "relative_path":str(path.relative_to(ROOT)), "filename":path.name,
        "extension":suffix, "file_size_bytes":path.stat().st_size,
        "row_count_excluding_header":row_count, "column_count":n_cols,
        "estimated_in_memory_bytes":memory_estimate, "inspection_status":read_note
    })
file_summary = pd.DataFrame(file_summary)
schema_summary = pd.DataFrame(schema_summary)
file_summary_path = ROOT/"artifacts/dataset_file_summary.csv"
schema_summary_path = ROOT/"artifacts/dataset_schema_summary.csv"
file_summary.to_csv(file_summary_path,index=False); schema_summary.to_csv(schema_summary_path,index=False)
display(file_summary.sort_values("file_size_bytes",ascending=False).head(20))
display(schema_summary.head(40))
print(f"Saved table: {file_summary_path}")
print(f"Saved table: {schema_summary_path}")
print(f"Available RAM: {environment['available_ram_gib']:.2f} GiB; largest estimated CSV materialization: {file_summary.estimated_in_memory_bytes.max()/2**30:.2f} GiB")

,relative_path,filename,extension,file_size_bytes,row_count_excluding_header,column_count,estimated_in_memory_bytes,inspection_status
1951,dataset/raw/extracted/data/metadata/answer_metada...,answer_metadata_task_1_2.csv,.csv,1019436477,19834820.0,6.0,1.408796e+09,sampled
1968,dataset/raw/extracted/data/train_data/train_task_...,train_task_1_2.csv,.csv,430049940,15867850.0,6.0,7.620757e+08,sampled
1972,dataset/raw/extracted/starter_kit/submission_temp...,submission_task_1_2.csv,.csv,109256805,3966963.0,4.0,1.270475e+08,sampled
1973,dataset/raw/extracted/starter_kit/submission_temp...,submission_task_1_2_old.csv,.csv,75775461,3966963.0,3.0,9.531184e+07,sampled
1952,dataset/raw/extracted/data/metadata/answer_metada...,answer_metadata_task_3_4.csv,.csv,73137422,1508917.0,6.0,1.071729e+08,sampled
1961,dataset/raw/extracted/data/test_data/test_private...,test_private_answers_task_2.csv,.csv,45822663,1983482.0,4.0,6.352379e+07,sampled
1960,dataset/raw/extracted/data/test_data/test_private...,test_private_answers_task_1.csv,.csv,45822661,1983482.0,4.0,6.352379e+07,sampled
1965,dataset/raw/extracted/data/test_data/test_public_...,test_public_answers_task_2.csv,.csv,45821599,1983481.0,4.0,6.352376e+07,sampled
1964,dataset/raw/extracted/data/test_data/test_public_...,test_public_answers_task_1.csv,.csv,45821597,1983481.0,4.0,6.352376e+07,sampled
1969,dataset/raw/extracted/data/train_data/train_task_...,train_task_3_4.csv,.csv,31731601,1382727.0,6.0,6.640740e+07,sampled


,file,column,sample_dtype,sample_non_null,sample_unique,example_values
0,dataset/raw/extracted/data/metadata/answer_metada...,AnswerId,float64,5000,5000,11808339.0 | 98649.0 | 259238.0
1,dataset/raw/extracted/data/metadata/answer_metada...,DateAnswered,str,5000,4963,2020-03-17 07:55:00.000 | 2018-12-18 14:23:00....
2,dataset/raw/extracted/data/metadata/answer_metada...,Confidence,float64,424,5,75.0 | 100.0 | 100.0
3,dataset/raw/extracted/data/metadata/answer_metada...,GroupId,int64,5000,2758,4186 | 9427 | 7651
4,dataset/raw/extracted/data/metadata/answer_metada...,QuizId,int64,5000,2124,14854 | 16895 | 1127
5,dataset/raw/extracted/data/metadata/answer_metada...,SchemeOfWorkId,float64,2461,45,28237.0 | 8386.0 | 40960.0
6,dataset/raw/extracted/data/metadata/answer_metada...,AnswerId,int64,5000,5000,1451945 | 45325 | 687013
7,dataset/raw/extracted/data/metadata/answer_metada...,DateAnswered,str,5000,4856,2019-10-30 14:34:00.000 | 2020-01-06 18:53:00....
8,dataset/raw/extracted/data/metadata/answer_metada...,Confidence,float64,1256,5,75.0 | 100.0 | 0.0
9,dataset/raw/extracted/data/metadata/answer_metada...,GroupId,int64,5000,316,4 | 185 | 235


Saved table: <repository_root>/artifacts/dataset_file_summary.csv
Saved table: <repository_root>/artifacts/dataset_schema_summary.csv
Available RAM: 3.10 GiB; largest estimated CSV materialization: 1.31 GiB


## 5. Full interaction and metadata quality validation

Determine whether the labeled interaction evidence is fit for temporal student-response prediction. Locate files by filename pattern, use explicit compact dtypes for the primary table, stream the large answer metadata, and validation grain, targets, identifiers, repeats, timestamps, confidence, demographics, and hierarchy integrity.

Leakage control: The validation measures target quality but creates no predictive features; post-response fields such as current confidence are flagged rather than used.

In [6]:
def locate_one(pattern, required=True):
    matches = sorted(p for p in (ROOT/"dataset/raw/extracted").rglob(pattern) if not p.name.startswith("._") and "__MACOSX" not in p.parts)
    if required and not matches:
        raise FileNotFoundError(f"Required raw file pattern not found: {pattern}")
    if len(matches)>1:
        print(f"Pattern {pattern} matched {len(matches)} files; using {matches[0]}")
    return matches[0] if matches else None

def match_column(columns, candidates):
    lookup={str(c).lower().replace("_",""):c for c in columns}
    for candidate in candidates:
        key=candidate.lower().replace("_","")
        if key in lookup: return lookup[key]
    return None

train_path = locate_one("train_task_1_2.csv")
question_path = locate_one("question_metadata_task_1_2.csv")
student_path = locate_one("student_metadata_task_1_2.csv")
answer_path = locate_one("answer_metadata_task_1_2.csv")
subject_path = locate_one("subject_metadata.csv")
public_answers_path = locate_one("test_public_answers_task_1.csv", required=False)
private_answers_path = locate_one("test_private_answers_task_1.csv", required=False)

train_header = pd.read_csv(train_path,nrows=0).columns.tolist()
train_dtypes={c:"Int32" for c in train_header if c.lower() in {"questionid","userid","answerid","answervalue","correctanswer","iscorrect"}}
train = pd.read_csv(train_path, dtype=train_dtypes, low_memory=False)
print(f"Loaded typed primary interaction table: {len(train):,} rows, {train.memory_usage(deep=True).sum()/2**30:.2f} GiB")
display(train.head())

student_meta=pd.read_csv(student_path,low_memory=False)
question_meta=pd.read_csv(question_path,low_memory=False)
subject_meta=pd.read_csv(subject_path,low_memory=False)
print(f"Metadata rows — students={len(student_meta):,}, questions={len(question_meta):,}, subjects={len(subject_meta):,}")

uid=match_column(train.columns,["UserId"]); qid=match_column(train.columns,["QuestionId"])
aid=match_column(train.columns,["AnswerId"]); target=match_column(train.columns,["IsCorrect"])
answer_value=match_column(train.columns,["AnswerValue"]); correct_answer=match_column(train.columns,["CorrectAnswer"])
if not all([uid,qid,aid,target]): raise ValueError(f"Primary schema missing required identifiers/target: {train.columns.tolist()}")

quality_rows=[]
def add_quality(check, value, status, severity, interpretation):
    quality_rows.append({"check":check,"value":value,"status":status,"severity":severity,"interpretation":interpretation})
add_quality("interaction_rows",len(train),"observed","info","Primary student-question interaction count")
add_quality("unique_students",train[uid].nunique(),"observed","info","Student coverage")
add_quality("unique_questions",train[qid].nunique(),"observed","info","Question coverage")
add_quality("unique_answers",train[aid].nunique(),"pass" if train[aid].is_unique else "review","high" if not train[aid].is_unique else "info","AnswerId should identify an interaction")
add_quality("impossible_target_values",int((~train[target].isin([0,1]) & train[target].notna()).sum()),"pass" if train[target].dropna().isin([0,1]).all() else "fail","critical","Binary target domain")
if answer_value:
    invalid_answer=int((~train[answer_value].isin([1,2,3,4]) & train[answer_value].notna()).sum())
    add_quality("invalid_answer_values",invalid_answer,"pass" if invalid_answer==0 else "review","medium","Eedi four-option answer values must be 1–4")

exact_duplicates=int(train.duplicated().sum())
repeated_pairs=int(train.duplicated([uid,qid],keep=False).sum())
unique_pairs=int(train[[uid,qid]].drop_duplicates().shape[0])
add_quality("exact_duplicate_rows",exact_duplicates,"pass" if exact_duplicates==0 else "review","high","Exact duplicates can distort interaction weighting")
add_quality("rows_in_repeated_student_question_pairs",repeated_pairs,"observed","medium","Repeated attempts require chronological handling, not silent removal")

# Stream answer metadata to avoid a second large materialization.
answer_header=pd.read_csv(answer_path,nrows=0).columns.tolist()
answer_id_col=match_column(answer_header,["AnswerId"]); date_col=match_column(answer_header,["DateAnswered"])
confidence_col=match_column(answer_header,["Confidence"]); group_col=match_column(answer_header,["GroupId"])
quiz_col=match_column(answer_header,["QuizId"]); scheme_col=match_column(answer_header,["SchemeOfWorkId"])
usecols=[c for c in [answer_id_col,date_col,confidence_col,group_col,quiz_col,scheme_col] if c]
answer_rows=0; answer_ids=set(); groups=set(); quizzes=set(); schemes=set(); date_min=None; date_max=None
missing_counts=Counter(); invalid_confidence=0; future_timestamps=0
now_utc=pd.Timestamp.now(tz="UTC")
for chunk in pd.read_csv(answer_path,usecols=usecols,chunksize=750_000,low_memory=False):
    answer_rows += len(chunk)
    for c in usecols: missing_counts[c] += int(chunk[c].isna().sum())
    if answer_id_col: answer_ids.update(chunk[answer_id_col].dropna().astype("int64").tolist())
    if group_col: groups.update(chunk[group_col].dropna().astype(str).tolist())
    if quiz_col: quizzes.update(chunk[quiz_col].dropna().astype(str).tolist())
    if scheme_col: schemes.update(chunk[scheme_col].dropna().astype(str).tolist())
    if confidence_col:
        numeric=pd.to_numeric(chunk[confidence_col],errors="coerce")
        invalid_confidence += int(((numeric<0)|(numeric>100)).sum())
    if date_col:
        dates=pd.to_datetime(chunk[date_col],errors="coerce",utc=True)
        if dates.notna().any():
            cmin,cmax=dates.min(),dates.max()
            date_min=cmin if date_min is None or cmin<date_min else date_min
            date_max=cmax if date_max is None or cmax>date_max else date_max
            future_timestamps += int((dates>now_utc).sum())
print(f"Streamed answer metadata: {answer_rows:,} rows")

add_quality("answer_metadata_rows",answer_rows,"observed","info","Response-context coverage")
add_quality("answer_id_train_coverage",round(train[aid].isin(answer_ids).mean(),6),"pass" if train[aid].isin(answer_ids).mean()>0.999 else "review","high","Join coverage from interactions to timestamps/context")
add_quality("unique_groups",len(groups),"observed","info","Group/class contexts")
add_quality("unique_quizzes",len(quizzes),"observed","info","Quiz contexts")
add_quality("unique_schemes_of_work",len(schemes),"observed","info","Scheme-of-work contexts")
add_quality("future_timestamps",future_timestamps,"pass" if future_timestamps==0 else "review","high","Future dates relative to execution timestamp")
add_quality("impossible_confidence_values",invalid_confidence,"pass" if invalid_confidence==0 else "review","medium","Confidence expected within recorded scale")

# Student demographics and ages use flexible actual-column detection.
student_uid=match_column(student_meta.columns,["UserId"]); dob_col=match_column(student_meta.columns,["DateOfBirth","BirthDate"])
gender_col=match_column(student_meta.columns,["Gender"]); premium_col=match_column(student_meta.columns,["PremiumPupil","PupilPremium"])
inconsistent_birth_dates=0; age_summary={}
if dob_col:
    dob=pd.to_datetime(student_meta[dob_col],errors="coerce",utc=True)
    inconsistent_birth_dates=int(((dob>now_utc)|(dob<pd.Timestamp("1900-01-01",tz="UTC"))).sum())
    if date_max is not None:
        ages=(date_max-dob).dt.total_seconds()/(365.2425*86400)
        age_summary={"age_min":float(ages.min()),"age_median":float(ages.median()),"age_max":float(ages.max()),"invalid_age_count":int(((ages<4)|(ages>25)).sum())}
add_quality("inconsistent_birth_dates",inconsistent_birth_dates,"pass" if inconsistent_birth_dates==0 else "review","medium","Birth date plausibility")

# Subject hierarchy integrity with actual names.
subj_id=match_column(subject_meta.columns,["SubjectId"]); parent_id=match_column(subject_meta.columns,["ParentId"])
subject_orphans=0; subject_cycles=0
if subj_id and parent_id:
    normalized_subject_ids=pd.to_numeric(subject_meta[subj_id],errors="coerce").astype("Int64")
    normalized_parent_ids=pd.to_numeric(subject_meta[parent_id],errors="coerce").astype("Int64")
    subject_ids=set(normalized_subject_ids.dropna().astype(int))
    parents=normalized_parent_ids.dropna().astype(int)
    subject_orphans=int((~parents.isin(subject_ids)).sum())
    parent_map={int(child):int(parent) for child,parent in zip(normalized_subject_ids,normalized_parent_ids) if pd.notna(child) and pd.notna(parent)}
    for node in parent_map:
        seen=set(); cur=node
        while cur in parent_map:
            if cur in seen: subject_cycles+=1; break
            seen.add(cur); cur=parent_map[cur]
add_quality("subject_orphan_parents",subject_orphans,"pass" if subject_orphans==0 else "review","high","Hierarchy parent referential integrity")
add_quality("subject_cycle_nodes",subject_cycles,"pass" if subject_cycles==0 else "review","high","Hierarchy must be acyclic")

quality_summary=pd.DataFrame(quality_rows)
missing_rows=[]
for file_name, frame in [(train_path.name,train),(student_path.name,student_meta),(question_path.name,question_meta),(subject_path.name,subject_meta)]:
    for c in frame.columns:
        missing_rows.append({"file":file_name,"column":c,"missing_count":int(frame[c].isna().sum()),"missing_rate":float(frame[c].isna().mean())})
for c in usecols:
    missing_rows.append({"file":answer_path.name,"column":c,"missing_count":missing_counts[c],"missing_rate":missing_counts[c]/max(answer_rows,1)})
missingness_summary=pd.DataFrame(missing_rows)
duplicate_summary=pd.DataFrame([
    {"scope":"primary_interactions","duplicate_definition":"exact full row","duplicate_rows":exact_duplicates,"rate":exact_duplicates/len(train)},
    {"scope":"primary_interactions","duplicate_definition":"repeated UserId-QuestionId pair (all involved rows)","duplicate_rows":repeated_pairs,"rate":repeated_pairs/len(train)},
    {"scope":"primary_interactions","duplicate_definition":"non-unique AnswerId rows","duplicate_rows":int(train.duplicated([aid],keep=False).sum()),"rate":float(train.duplicated([aid],keep=False).mean())}
])
student_counts=train.groupby(uid,observed=True).size().rename("interaction_count")
question_counts=train.groupby(qid,observed=True).size().rename("interaction_count")
student_activity_summary=student_counts.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).rename("value").reset_index().rename(columns={"index":"statistic"})
question_activity_summary=question_counts.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).rename("value").reset_index().rename(columns={"index":"statistic"})
target_distribution_summary=(train[target].value_counts(dropna=False).rename_axis("IsCorrect").reset_index(name="count"))
target_distribution_summary["proportion"]=target_distribution_summary["count"]/len(train)
subject_hierarchy_summary=pd.DataFrame([{
    "subject_rows":len(subject_meta),"unique_subjects":subject_meta[subj_id].nunique() if subj_id else np.nan,
    "root_subjects":int(subject_meta[parent_id].isna().sum()) if parent_id else np.nan,
    "orphan_parent_references":subject_orphans,"cycle_nodes_detected":subject_cycles
}])

outputs={
"dataset_quality_summary.csv":quality_summary,"missingness_summary.csv":missingness_summary,
"duplicate_summary.csv":duplicate_summary,"student_activity_summary.csv":student_activity_summary,
"question_activity_summary.csv":question_activity_summary,"subject_hierarchy_summary.csv":subject_hierarchy_summary,
"target_distribution_summary.csv":target_distribution_summary}
for name,frame in outputs.items():
    path=ROOT/"artifacts"/name; frame.to_csv(path,index=False); print(f"Saved table: {path}")
display(quality_summary)
display(target_distribution_summary)
display(duplicate_summary)
display(student_activity_summary)
display(question_activity_summary)
display(subject_hierarchy_summary)
print(f"Date coverage: {date_min} to {date_max}; age validation: {age_summary}")

Loaded typed primary interaction table: 15,867,850 rows, 0.44 GiB


,QuestionId,UserId,AnswerId,IsCorrect,CorrectAnswer,AnswerValue
0,16997,65967,12453206,0,4,2
1,16531,62121,15686710,1,1,1
2,15911,50013,13598796,0,3,1
3,1701,104909,10511925,0,4,3
4,22896,21748,941747,0,1,4


Metadata rows — students=118,971, questions=27,613, subjects=388


Streamed answer metadata: 19,834,820 rows


Saved table: <repository_root>/artifacts/dataset_quality_summary.csv
Saved table: <repository_root>/artifacts/missingness_summary.csv
Saved table: <repository_root>/artifacts/duplicate_summary.csv
Saved table: <repository_root>/artifacts/student_activity_summary.csv
Saved table: <repository_root>/artifacts/question_activity_summary.csv
Saved table: <repository_root>/artifacts/subject_hierarchy_summary.csv
Saved table: <repository_root>/artifacts/target_distribution_summary.csv


...

,scope,duplicate_definition,duplicate_rows,rate
0,primary_interactions,exact full row,0,0.0
1,primary_interactions,repeated UserId-QuestionId pair (all involved ...,0,0.0
2,primary_interactions,non-unique AnswerId rows,0,0.0


,statistic,value
0,count,118971.000000
1,mean,133.375781
2,std,127.762689
3,min,29.000000
4,1%,38.000000
5,5%,42.000000
6,25%,57.000000
7,50%,88.000000
8,75%,159.000000
9,95%,380.000000


,statistic,value
0,count,27613.000000
1,mean,574.651432
2,std,654.449058
3,min,34.000000
4,1%,41.000000
5,5%,49.000000
6,25%,109.000000
7,50%,281.000000
8,75%,850.000000
9,95%,1942.000000


,subject_rows,unique_subjects,root_subjects,orphan_parent_references,cycle_nodes_detected
0,388,388,2,0,0


Date coverage: 2018-09-01 00:53:00+00:00 to 2020-05-02 11:14:00+00:00; age validation: {'age_min': -7979.579408049294, 'age_median': 14.501237001596351, 'age_max': 263.66720207959247, 'invalid_age_count': 1933}


## 6. Official holdout overlap validation

Determine whether the supplementary competition holdouts share entities with labeled training data. Discover all Task 1 public/private CSVs, inspect actual columns, and compute overlap wherever student/question identifiers are present; answer-label files alone are explicitly reported as insufficient for entity-overlap estimation.

Leakage control: Official labels remain evaluation-only and are never used for tuning.

In [7]:
official_files=sorted({p for pat in ["*public*task_1*.csv","*private*task_1*.csv"] for p in (ROOT/"dataset/raw/extracted").rglob(pat) if not p.name.startswith("._") and "__MACOSX" not in p.parts})
overlap_rows=[]
for path in official_files:
    frame=pd.read_csv(path,low_memory=False)
    ouid=match_column(frame.columns,["UserId"]); oqid=match_column(frame.columns,["QuestionId"]); oaid=match_column(frame.columns,["AnswerId"])
    overlap_rows.append({
        "file":str(path.relative_to(ROOT)),"rows":len(frame),"columns":" | ".join(frame.columns.astype(str)),
        "student_overlap_rate":float(frame[ouid].isin(train[uid]).mean()) if ouid else np.nan,
        "question_overlap_rate":float(frame[oqid].isin(train[qid]).mean()) if oqid else np.nan,
        "answer_id_overlap_rate":float(frame[oaid].isin(train[aid]).mean()) if oaid else np.nan,
        "entity_overlap_available":bool(ouid or oqid),
        "notes":"Direct entity IDs present" if (ouid or oqid) else "Answer-label file does not expose UserId/QuestionId directly"
    })
official_overlap=pd.DataFrame(overlap_rows)
display(official_overlap)
print("Official holdout labels were inspected only for schema and overlap; no result is used for model selection.")

,file,rows,columns,student_overlap_rate,question_overlap_rate,answer_id_overlap_rate,entity_overlap_available,notes
0,dataset/raw/extracted/data/test_data/test_private...,1983482,QuestionId | UserId | AnswerId | IsCorrect,1.0,1.0,0.0,True,Direct entity IDs present
1,dataset/raw/extracted/data/test_data/test_public_...,1983481,QuestionId | UserId | AnswerId | IsCorrect,1.0,1.0,0.0,True,Direct entity IDs present


Official holdout labels were inspected only for schema and overlap; no result is used for model selection.


## 7. Frozen research scope, split protocol, and leakage controls

Pre-register the prediction question and generalization scenarios before feature engineering or model comparison. Define temporal within-student, unseen-student, unseen-question, short-history, and official supplementary evaluations; list high-risk leakage paths and enforce train-only/shifted construction.

Leakage control: This is the controlling protocol for notebooks 02–06; later code must fail checks if entity or temporal boundaries are violated.

In [8]:
research_scope_summary=pd.DataFrame([
{"item":"Primary unit","definition":"One student-question interaction at a timestamp"},
{"item":"Target","definition":"IsCorrect: 1 correct, 0 incorrect"},
{"item":"Primary objective","definition":"Estimate next-response correctness from metadata available at assignment time and outcomes observed strictly before prediction"},
{"item":"Primary selection metric","definition":"Validation ROC-AUC under the main temporal split"},
{"item":"Co-primary metrics","definition":"Log Loss, Brier Score, Expected Calibration Error"},
{"item":"Decision layer","definition":"Exploratory non-causal offline risk stratification; not a validated recommendation policy"},
{"item":"Evidence boundary","definition":"Predictive observational study; no causal learning-gain or intervention-effect claim"}
])
split_protocol_summary=pd.DataFrame([
{"scenario":"A","split_id":"train / validation / temporal_test","entity_rule":"Eligible non-external students/questions","time_rule":"Earliest ~70% / next ~15% / latest ~15% per student","selection_use":"Train and validation only"},
{"scenario":"B","split_id":"external_unseen_student","entity_rule":"Deterministic complete student holdout balanced by available strata","time_rule":"All interactions held out","selection_use":"Secondary learner-disjoint stress evaluation; not confirmatory"},
{"scenario":"C","split_id":"external_unseen_question","entity_rule":"Deterministic complete question holdout preserving subject diversity","time_rule":"Held-out question outcomes never update predictor state","selection_use":"Secondary item-disjoint stress evaluation; not confirmatory"},
{"scenario":"D","split_id":"stress_short_history","entity_rule":"Eligible temporal-test students","time_rule":"Evaluate history caps 5, 10, 20, 50","selection_use":"Stress evaluation only"},
{"scenario":"E","split_id":"official_public_test / official_private_test","entity_rule":"Official competition holdouts overlap training entities","time_rule":"As supplied","selection_use":"Supplementary schema/overlap evidence only; not independent confirmation"}
])
leakage_control_summary=pd.DataFrame([
{"risk":"Current IsCorrect in historical features","severity":"Critical","control":"All rolling/sequence outcomes are shifted by one; assertions compare source position"},
{"risk":"Target encoding without out-of-fold construction","severity":"Critical","control":"Training target encodings are chronological or out-of-fold; validation/test maps are train-only"},
{"risk":"Full-data student/question averages","severity":"Critical","control":"Chronological past-only train features with train-fitted fallback maps"},
{"risk":"Future interactions used for current prediction","severity":"Critical","control":"Stable sort by DateAnswered and AnswerId; cumulative state updated after feature emission"},
{"risk":"Random interaction split","severity":"High","control":"Per-student temporal split plus entity-disjoint external tests"},
{"risk":"Current-answer confidence leakage","severity":"Critical","control":"Only lagged prior confidence is eligible; current confidence is excluded"},
{"risk":"Repeated student-question attempts","severity":"High","control":"Retain attempts in chronological order and mark repeat count; never leak later attempts"},
{"risk":"Official test entity overlap","severity":"High","control":"Official files are supplementary only and never used for tuning"},
{"risk":"Subject statistics from test labels","severity":"Critical","control":"Subject mappings are trained on training labels only with hierarchy/global fallback"},
{"risk":"Demographic-only recommendation","severity":"High","control":"Protected metadata is diagnostic/contextual only; never the sole recommendation basis"}
])
for name,frame in {"research_scope_summary.csv":research_scope_summary,"split_protocol_summary.csv":split_protocol_summary,"leakage_control_summary.csv":leakage_control_summary}.items():
    path=ROOT/"artifacts"/name; frame.to_csv(path,index=False); print(f"Saved table: {path}")
display(research_scope_summary); display(split_protocol_summary); display(leakage_control_summary)

config={
"project_name":"AdaptiveMath-AI","seed":SEED,"timezone":"UTC for computation; source timestamps retained",
"primary_task":"temporal binary student response prediction","target":"IsCorrect","positive_class":1,
"selection_metric":"validation temporal ROC-AUC","co_primary_metrics":["Log Loss","Brier Score","Expected Calibration Error"],
"split_fractions":{"train":0.70,"validation":0.15,"temporal_test":0.15},
"external_holdout_fractions":{"students":0.05,"questions":0.05},
"minimum_history":5,"short_history_caps":[5,10,20,50],"sequence_lengths":[30],
"cohort_policy":{"max_students":12000,"max_interactions":600000,"minimum_student_interactions":20,
                 "selection":"deterministic stratified hash/activity selection before model comparison",
                 "same_cohort_all_comparable_models":True},
"deep_learning":{"device":"mps" if mps_available else "cpu","max_epochs":4,"early_stopping_patience":2,"batch_size":512,"mixed_precision":False,"comparison_scope":"bounded mechanism-proxy screening; not canonical SOTA reproduction"},
"anti_overfitting":{"test_tuning":False,"external_tuning":False,"official_label_tuning":False,"confirmatory_status":"No pristine confirmatory test remains after earlier development; all holdout results are explicitly secondary"},
"raw_file_paths":{k:str(v.relative_to(ROOT)) for k,v in {"train":train_path,"answers":answer_path,"students":student_path,"questions":question_path,"subjects":subject_path}.items()},
"environment":environment
}
config_path=ROOT/"artifacts/experiment_config.json"; config_path.write_text(json.dumps(config,indent=2,default=str),encoding="utf-8")
print(f"Saved configuration: {config_path} ({config_path.stat().st_size/1024:.1f} KiB)")
display(Markdown("```json\n"+json.dumps(config,indent=2,default=str)[:3500]+"\n```"))

Saved table: <repository_root>/artifacts/research_scope_summary.csv
Saved table: <repository_root>/artifacts/split_protocol_summary.csv
Saved table: <repository_root>/artifacts/leakage_control_summary.csv


,item,definition
0,English title,AdaptiveMath-AI: Leakage-Controlled Prequentia...
1,Russian title,AdaptiveMath-AI: прогнозирование следующего от...
2,Primary unit,One student-question interaction at a timestamp
3,Target,"IsCorrect: 1 correct, 0 incorrect"
4,Primary objective,Estimate next-response correctness from metada...
5,Primary selection metric,Validation ROC-AUC under the main temporal split
6,Co-primary metrics,"Log Loss, Brier Score, Expected Calibration Error"
7,Decision layer,Exploratory non-causal offline risk stratifica...
8,Evidence boundary,Predictive observational study; no causal lear...


,scenario,split_id,entity_rule,time_rule,selection_use
0,A,train / validation / temporal_test,Eligible non-external students/questions,Earliest ~70% / next ~15% / latest ~15% per st...,Train and validation only
1,B,external_unseen_student,Deterministic complete student holdout balance...,All interactions held out,Secondary learner-disjoint stress evaluation; ...
2,C,external_unseen_question,Deterministic complete question holdout preser...,Held-out question outcomes never update predic...,Secondary item-disjoint stress evaluation; not...
3,D,stress_short_history,Eligible temporal-test students,"Evaluate history caps 5, 10, 20, 50",Stress evaluation only
4,E,official_public_test / official_private_test,Official competition holdouts overlap training...,As supplied,Supplementary schema/overlap evidence only; no...


,risk,severity,control
0,Current IsCorrect in historical features,Critical,All rolling/sequence outcomes are shifted by o...
1,Target encoding without out-of-fold construction,Critical,Training target encodings are chronological or...
2,Full-data student/question averages,Critical,Chronological past-only train features with tr...
3,Future interactions used for current prediction,Critical,Stable sort by DateAnswered and AnswerId; cumu...
4,Random interaction split,High,Per-student temporal split plus entity-disjoin...
5,Current-answer confidence leakage,Critical,Only lagged prior confidence is eligible; curr...
6,Repeated student-question attempts,High,Retain attempts in chronological order and mar...
7,Official test entity overlap,High,Official files are supplementary only and neve...
8,Subject statistics from test labels,Critical,Subject mappings are trained on training label...
9,Demographic-only recommendation,High,Protected metadata is diagnostic/contextual on...


Saved configuration: <repository_root>/artifacts/experiment_config.json (3.0 KiB)


```json
{
  "project_name": "AdaptiveMath-AI",
  "seed": 20260713,
  "timezone": "UTC for computation; source timestamps retained",
  "titles": {
    "english": "AdaptiveMath-AI: Leakage-Controlled Prequential Student-Response Prediction with Hierarchical Item-Residual Memory",
    "russian": "AdaptiveMath-AI: \u043f\u0440\u043e\u0433\u043d\u043e\u0437\u0438\u0440\u043e\u0432\u0430\u043d\u0438\u0435 \u0441\u043b\u0435\u0434\u0443\u044e\u0449\u0435\u0433\u043e \u043e\u0442\u0432\u0435\u0442\u0430 \u0443\u0447\u0430\u0449\u0435\u0433\u043e\u0441\u044f \u0441 \u043a\u043e\u043d\u0442\u0440\u043e\u043b\u0435\u043c \u0443\u0442\u0435\u0447\u043a\u0438 \u0438 \u0438\u0435\u0440\u0430\u0440\u0445\u0438\u0447\u0435\u0441\u043a\u043e\u0439 residual-\u043f\u0430\u043c\u044f\u0442\u044c\u044e \u0437\u0430\u0434\u0430\u043d\u0438\u0439"
  },
  "primary_task": "temporal binary student response prediction",
  "target": "IsCorrect",
  "positive_class": 1,
  "selection_metric": "validation temporal ROC-AUC",
  "co_primary_metrics": [
    "Log Loss",
    "Brier Score",
    "Expected Calibration Error"
  ],
  "split_fractions": {
    "train": 0.7,
    "validation": 0.15,
    "temporal_test": 0.15
  },
  "external_holdout_fractions": {
    "students": 0.05,
    "questions": 0.05
  },
  "minimum_history": 5,
  "short_history_caps": [
    5,
    10,
    20,
    50
  ],
  "sequence_lengths": [
    30
  ],
  "cohort_policy": {
    "max_students": 12000,
    "max_interactions": 600000,
    "minimum_student_interactions": 20,
    "selection": "deterministic stratified hash/activity selection before model comparison",
    "same_cohort_all_comparable_models": true
  },
  "deep_learning": {
    "device": "mps",
    "max_epochs": 4,
    "early_stopping_patience": 2,
    "batch_size": 512,
    "mixed_precision": false,
    "comparison_scope": "bounded mechanism-proxy screening; not canonical SOTA reproduction"
  },
  "anti_overfitting": {
    "test_tuning": false,
    "external_tuning": false,
    "official_label_tuning": false,
    "confirmatory_status": "No pristine confirmatory test remains after earlier development; all holdout results are explicitly secondary"
  },
  "raw_file_paths": {
    "train": "dataset/raw/extracted/data/train_data/train_task_1_2.csv",
    "answers": "dataset/raw/extracted/data/metadata/answer_metadata_task_1_2.csv",
    "students": "dataset/raw/extracted/data/metadata/student_metadata_task_1_2.csv",
    "questions": "dataset/raw/extracted/data/metadata/question_metadata_task_1_2.csv",
    "subjects": "dataset/raw/extracted/data/metadata/subject_metadata.csv"
  },
  "environment": {
    "project_root": "<repository_root>",
    "python": "3.11.15",
    "platform": "macOS-26.6-arm64-arm-64bit",
    "machine": "arm64",
    "cpu_count_logical": 10,
    "cpu_count_physical": 10,
    "available_ram_gib": 3.099822998046875,
    "total_ram_gib": 16.0,
    "free_disk_gib": 280.9512710571289,
    "torch_version": "2.11.0",
    "mps_available": true,
    "seed": 20260713,
    "execution_timestamp_utc": "2026-07-31T13:47:48.791728+00:00"
  }
}
```